In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VIDEO_DIR = DATA_DIR / "Video"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VIDEO_DIR =", VIDEO_DIR)
print("EXCEL_DIR =", EXCEL_DIR)
# ✅ Mets ici la racine où sont tes D01, D02, ...
ROOT = VIDEO_DIR

def compute_shoulder_width_from_pose_xlsx(pose_xlsx: Path):
    df = pd.read_excel(pose_xlsx)

    needed = ["LEFT_SHOULDER_x","LEFT_SHOULDER_y","RIGHT_SHOULDER_x","RIGHT_SHOULDER_y"]
    if not all(c in df.columns for c in needed):
        return None  # pas les bonnes colonnes

    sw = np.sqrt(
        (df["LEFT_SHOULDER_x"] - df["RIGHT_SHOULDER_x"])**2 +
        (df["LEFT_SHOULDER_y"] - df["RIGHT_SHOULDER_y"])**2
    )

    sw = sw.replace([np.inf, -np.inf], np.nan).dropna()
    if len(sw) == 0:
        return None

    return {
        "shoulder_width_median": float(sw.median()),
        "shoulder_width_mean": float(sw.mean()),
        "n_frames_used": int(len(sw)),
    }

pose_files = list(ROOT.rglob("*_pose.xlsx"))
print("Found pose files:", len(pose_files))

rows = []
for f in pose_files:
    res = compute_shoulder_width_from_pose_xlsx(f)
    if res is None:
        print("⚠️ Skip (missing cols/empty):", f)
        continue

    # infos utiles pour retrouver la vidéo / condition
    rel = f.relative_to(ROOT)
    parts = rel.parts  # ex: ("D01","P1","SEATED","video_pose.xlsx")
    D = parts[0] if len(parts) > 0 else ""
    P = parts[1] if len(parts) > 1 else ""
    condition = parts[2] if len(parts) > 2 else ""

    # souvent le nom vidéo = stem sans "_pose"
    # ex: "SEATEDD02_pose.xlsx" => "SEATEDD02"
    video_id = f.stem.replace("_pose", "")

    rows.append({
        "video_id": video_id,
        "D": D,
        "P": P,
        "condition": condition,
        "pose_xlsx_path": str(f),
        **res
    })

df_sw = pd.DataFrame(rows).sort_values(["D","P","condition","video_id"])
out_path = EXCEL_DIR / "shoulder_width_by_video.xlsx"
df_sw.to_excel(out_path, index=False)
print("✅ Saved:", out_path)

Found pose files: 120
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/shoulder_width_by_video.xlsx


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------
# 0) Chemins du projet
# --------------------------
CODE_DIR = Path().resolve()              # .../SYNCOGEST/Code/Codes MP
PROJECT_ROOT = CODE_DIR.parent.parent    # .../SYNCOGEST
DATA_DIR = PROJECT_ROOT / "DATA"

VIDEO_DIR = DATA_DIR / "Video"
EXCEL_DIR = DATA_DIR / "Excels_code"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("VIDEO_DIR =", VIDEO_DIR)
print("EXCEL_DIR =", EXCEL_DIR)

ROOT = VIDEO_DIR

# --- chemins ---
summary_path = EXCEL_DIR / "Summary_Motion_Nose_Wrists_equalizedFrames.xlsx"   # adapte si besoin
sw_path      = EXCEL_DIR / "shoulder_width_by_video.xlsx"

# --- charge ---
summary = pd.read_excel(summary_path)
sw = pd.read_excel(sw_path)

# =========================
# 1) Trouver la colonne vidéo dans summary
# =========================
# Essaie automatique (tu peux forcer à la main en mettant SUMMARY_VIDEO_COL = "video_id" par ex)
candidates = ["video_id", "video", "Video", "VideoName", "video_name", "file", "filename", "trial"]
SUMMARY_VIDEO_COL = None
for c in candidates:
    if c in summary.columns:
        SUMMARY_VIDEO_COL = c
        break

if SUMMARY_VIDEO_COL is None:
    raise ValueError(
        "Je ne trouve pas de colonne vidéo dans summary. "
        "Ajoute une colonne 'video_id' ou indique le nom exact de la colonne."
    )

print("✅ Colonne utilisée pour identifier la vidéo dans summary:", SUMMARY_VIDEO_COL)

# =========================
# 2) Construire une clé video_id compatible
# =========================
# sw contient déjà 'video_id'
# summary: si c'est 'file' avec extension (.mp4), on nettoie.
summary = summary.copy()

if SUMMARY_VIDEO_COL != "video_id":
    # crée une colonne video_id standardisée
    summary["video_id"] = summary[SUMMARY_VIDEO_COL].astype(str)

    # enlève extensions fréquentes
    for ext in [".mp4", ".mov", ".avi", ".mkv"]:
        summary["video_id"] = summary["video_id"].str.replace(ext, "", regex=False)

    # enlève éventuellement "_pose" si jamais ça traîne
    summary["video_id"] = summary["video_id"].str.replace("_pose", "", regex=False)
else:
    summary["video_id"] = summary["video_id"].astype(str)

# =========================
# 3) Merge largeur épaules -> summary (1 ligne = 1 participant)
# =========================
out = summary.merge(
    sw[["video_id", "shoulder_width_median", "shoulder_width_mean"]],
    on="video_id",
    how="left"
)
# 1) Trouver les lignes sans shoulder width
missing_rows = out.loc[out["shoulder_width_median"].isna()].copy()

print("Nombre de lignes manquantes :", len(missing_rows))

# 2) Voir les video_id concernés
missing_ids = sorted(missing_rows["video_id"].dropna().astype(str).unique())

print("\nVideo IDs sans shoulder width :")
for vid in missing_ids:
    print("-", repr(vid))

# 3) Comparer avec ce qu'il y a dans sw
sw_ids = set(sw["video_id"].astype(str).str.strip())
summary_ids = set(summary["video_id"].astype(str).str.strip())

missing_in_sw = sorted(summary_ids - sw_ids)

print("\nIDs présents dans summary mais absents de shoulder_width_by_video :")
for vid in missing_in_sw:
    print("-", repr(vid))

# contrôle : combien de lignes n'ont pas trouvé de shoulder width ?
n_missing = out["shoulder_width_median"].isna().sum()
print(f"⚠️ Lignes sans shoulder width trouvé: {n_missing}/{len(out)}")
if n_missing > 0:
    # affiche quelques exemples
    print(out.loc[out["shoulder_width_median"].isna(), ["video_id"]].head(10))

# =========================
# 4) Calculs QDM poignets + normalisation
# =========================
needed_cols = ["total_motion_LEFT_WRIST", "total_motion_RIGHT_WRIST"]
for c in needed_cols:
    if c not in out.columns:
        raise ValueError(f"Colonne manquante dans summary: {c}")

# total poignets
out["QDM_WRISTS"] = out["total_motion_LEFT_WRIST"] + out["total_motion_RIGHT_WRIST"]

# normalisation morpho par largeur épaules (médiane)
out["total_motion_LEFT_WRIST_norm"]  = out["total_motion_LEFT_WRIST"]  / out["shoulder_width_median"]
out["total_motion_RIGHT_WRIST_norm"] = out["total_motion_RIGHT_WRIST"] / out["shoulder_width_median"]
out["QDM_WRISTS_norm"]               = out["QDM_WRISTS"]               / out["shoulder_width_median"]

# (optionnel) si tu veux aussi garder un "moyen par membre"
out["QDM_WRISTS_norm_meanPerWrist"] = (out["total_motion_LEFT_WRIST_norm"] + out["total_motion_RIGHT_WRIST_norm"]) / 2

# =========================
# 5) Sauvegarde
# =========================
out_path = EXCEL_DIR / f"{summary_path.stem}_normByShoulders.xlsx"
out.to_excel(out_path, index=False)
print("✅ Saved:", out_path)

✅ Colonne utilisée pour identifier la vidéo dans summary: video_name
Nombre de lignes manquantes : 3

Video IDs sans shoulder width :
- 'Summary_Motion_Nose_Wrists_equalizedFrames'
- 'Summary_Motion_Nose_Wrists_equalizedFrames_normByShoulders'
- 'shoulder_width_by_video'

IDs présents dans summary mais absents de shoulder_width_by_video :
- 'Summary_Motion_Nose_Wrists_equalizedFrames'
- 'Summary_Motion_Nose_Wrists_equalizedFrames_normByShoulders'
- 'shoulder_width_by_video'
⚠️ Lignes sans shoulder width trouvé: 3/123
                                              video_id
120         Summary_Motion_Nose_Wrists_equalizedFrames
121  Summary_Motion_Nose_Wrists_equalizedFrames_nor...
122                            shoulder_width_by_video
✅ Saved: /Users/matysprecloux/Desktop/SYNCOGEST/DATA/Video/Summary_Motion_Nose_Wrists_equalizedFrames_normByShoulders.xlsx
